In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import (
    BertTokenizerFast,
    BertModel,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_curve, auc, roc_auc_score
)

SEED = 42
MAX_LEN = 256     
BATCH_SIZE = 16       
NUM_EPOCHS = 3       
LEARNING_RATE = 1e-5      
WARMUP_RATIO = 0.1    
DROPOUT_RATE = 0.4
WEIGHT_DECAY = 0.01 

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch версія: {torch.__version__}')
print(f'CUDA доступна: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'\nВикористовується: {device}')
print(f'\nПараметри тренування:')
print(f'   MAX_LEN     = {MAX_LEN}')
print(f'   BATCH_SIZE  = {BATCH_SIZE}')
print(f'   NUM_EPOCHS  = {NUM_EPOCHS}')
print(f'   LR          = {LEARNING_RATE}')

In [ ]:
DATA_PATH = 'C:/Users/igrew/OneDrive/Desktop/Course Work/Datasets/news_detector/last_dataset.csv'

df = pd.read_csv(DATA_PATH)
print(f'Розмір датасету: {df.shape}')
print(f'Колонки: {list(df.columns)}')
df.head()

In [ ]:
df = df.dropna(subset=['text', 'label'])
print(f'Розподіл міток:')
print(df['label'].value_counts())
print(f'\nПісля очищення: {len(df)} записів')

true_count = (df['label'] == 1).sum()
fake_count = (df['label'] == 0).sum()
print(f'TRUE (правдиві): {true_count}')
print(f'FAKE (фейкові):  {fake_count}')
df.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Аналіз датасету', fontsize=16, fontweight='bold')

# 1. Pie chart — розподіл класів
fake_count = (df['label'] == 0).sum()
true_count = (df['label'] == 1).sum()
axes[0, 0].pie(
    [fake_count, true_count],
    labels=[f'FAKE ({fake_count})', f'TRUE ({true_count})'],
    colors=['#e74c3c', '#2ecc71'],
    autopct='%1.1f%%',
    startangle=90,
    explode=(0.05, 0.05),
    shadow=True
)
axes[0, 0].set_title('Розподіл класів', fontweight='bold')

# 2. Bar chart — кількість по класах
axes[0, 1].bar(['FAKE', 'TRUE'], [fake_count, true_count],
               color=['#e74c3c', '#2ecc71'], edgecolor='black', alpha=0.85)
axes[0, 1].set_title('Кількість зразків по класах', fontweight='bold')
axes[0, 1].set_ylabel('Кількість')
for i, v in enumerate([fake_count, true_count]):
    axes[0, 1].text(i, v + 50, str(v), ha='center', fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Розподіл довжини текстів (символів)
df['text_len'] = df['text'].str.len()
for label, color, name in [(0, '#e74c3c', 'FAKE'), (1, '#2ecc71', 'TRUE')]:
    subset = df[df['label'] == label]['text_len']
    axes[1, 0].hist(subset, bins=50, alpha=0.6, color=color,
                    label=name, edgecolor='white')
axes[1, 0].set_title('Розподіл довжини текстів (символів)', fontweight='bold')
axes[1, 0].set_xlabel('Кількість символів')
axes[1, 0].set_ylabel('Частота')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Розподіл кількості слів
df['word_count'] = df['text'].str.split().str.len()
for label, color, name in [(0, '#e74c3c', 'FAKE'), (1, '#2ecc71', 'TRUE')]:
    subset = df[df['label'] == label]['word_count']
    axes[1, 1].hist(subset, bins=50, alpha=0.6, color=color,
                    label=name, edgecolor='white')
axes[1, 1].axvline(x=256, color='orange', linestyle='--',
                   linewidth=2, label='MAX_LEN=256')
axes[1, 1].set_title('Розподіл кількості слів', fontweight='bold')
axes[1, 1].set_xlabel('Кількість слів')
axes[1, 1].set_ylabel('Частота')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('dataset_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nСтатистика:')
print(f'FAKE: {fake_count} | TRUE: {true_count}')
print(df[['text_len', 'word_count']].describe().round(1))
pct_over = (df['word_count'] > 256).mean() * 100
print(f'Текстів довших за 256 слів: {pct_over:.1f}%')

In [ ]:
# Train / Val / Test split: 70% / 15% / 15%
X = df['text'].values
y = df['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f'Train:      {len(X_train):>6} зразків ({len(X_train)/len(X)*100:.1f}%)')
print(f'Validation: {len(X_val):>6} зразків ({len(X_val)/len(X)*100:.1f}%)')
print(f'Test:       {len(X_test):>6} зразків ({len(X_test)/len(X)*100:.1f}%)')

In [ ]:
MODEL_NAME = 'bert-base-multilingual-cased'
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)
print(f'Токенізатор завантажено!')
print(f'   Розмір словника: {tokenizer.vocab_size:,}')

sample = 'Це фейкова новина про українську армію'
tokens = tokenizer(sample, return_tensors='pt')
print(f'\nПриклад токенізації:')
print(f'   Текст: "{sample}"')
print(f'   Токени: {tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])}')
print(f'   IDs: {tokens["input_ids"][0].tolist()}')

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(label, dtype=torch.long)
        }


train_dataset = NewsDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = NewsDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_dataset  = NewsDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'DataLoader-и створено:')
print(f'   Train: {len(train_loader)} батчів × {BATCH_SIZE} = {len(train_dataset)} зразків')
print(f'   Val:   {len(val_loader)}  батчів')
print(f'   Test:  {len(test_loader)} батчів')

In [ ]:
# HEAD A: Простий лінійний класифікатор (Baseline)
class BertClassifierHeadA(nn.Module):
    """
    Head A — Baseline: один лінійний шар поверх [CLS] токену.
    """
    def __init__(self, bert_model_name, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden = self.bert.config.hidden_size  # 768
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] токен
        x = self.dropout(cls_output)
        logits = self.classifier(x)
        return logits

print('Head A визначено: Linear baseline')

In [ ]:
# HEAD B: MLP з BatchNormalization
class BertClassifierHeadB(nn.Module):
    """
    Head B — MLP з BatchNorm.
    
    Гіпотеза: BatchNorm стабілізує розподіл активацій між батчами,
    що допомагає при fine-tuning BERT з малим LR. Додатковий прихований
    шар дозволяє вивчити нелінійні комбінації ознак.
    
    Архітектура:
        [CLS] (768) → Linear(768→512) → BatchNorm1d → ReLU → Dropout
                    → Linear(512→2)
    """
    def __init__(self, bert_model_name, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden = self.bert.config.hidden_size  # 768

        self.classifier = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] токен
        logits = self.classifier(cls_output)
        return logits

print('Head B визначено: MLP + BatchNorm')

In [ ]:
# HEAD C: Attention Pooling замість [CLS]
class AttentionPooling(nn.Module):
    """
    Замість одного [CLS] токену — зважене усереднення всіх токенів.
    Модель сама навчається, яким токенам приділяти більше уваги.
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states, attention_mask):
        scores = self.attention(hidden_states).squeeze(-1)
        scores = scores.masked_fill(attention_mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        pooled = (hidden_states * weights).sum(dim=1)       
        return pooled


class BertClassifierHeadC(nn.Module):
    """
    Head C — Attention Pooling + MLP.
    
    Гіпотеза: [CLS] токен не завжди оптимально агрегує інформацію
    для довгих текстів. Attention Pooling дозволяє моделі самостійно
    визначати, які слова є найбільш інформативними для класифікації.
    
    Архітектура:
        Всі токени (768) → AttentionPooling → Linear(768→256)
                         → LayerNorm → ReLU → Dropout → Linear(256→2)
    """
    def __init__(self, bert_model_name, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden = self.bert.config.hidden_size  # 768

        self.attention_pool = AttentionPooling(hidden)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        all_hidden = outputs.last_hidden_state   # (batch, seq_len, 768)
        pooled = self.attention_pool(all_hidden, attention_mask)
        logits = self.classifier(pooled)
        return logits

print('Head C визначено: Attention Pooling')
print('\nПідсумок архітектур:')
print('   Head A: BERT → [CLS] → Dropout → Linear(768→2)')
print('   Head B: BERT → [CLS] → Linear(768→512) → BN → ReLU → Dropout → Linear(512→2)')
print('   Head C: BERT → AttPool → Linear(768→256) → LN → ReLU → Dropout → Linear(256→2)')

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc='  Train', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, acc, f1


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, acc, f1, np.array(all_labels), np.array(all_preds), np.array(all_probs)


def train_model(model, train_loader, val_loader, num_epochs, lr, device, model_name='model'):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'train_f1': [], 'val_f1': []}
    best_val_f1 = 0
    best_model_path = f'{model_name}_best.pt'

    print(f'\nТренування {model_name}...')
    print(f'   Кроків warmup: {warmup_steps} / {total_steps}')
    print('=' * 75)
    print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>7} | {"Val F1":>6}')
    print('=' * 75)

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
        val_loss, val_acc, val_f1, _, _, _ = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)

        flag = ''
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            flag = ' SAVED'

        print(f'{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.4f} | {val_loss:>8.4f} | {val_acc:>7.4f} | {val_f1:>6.4f}{flag}')

    print('=' * 75)
    print(f'Найкращий Val F1: {best_val_f1:.4f}')

    model.load_state_dict(torch.load(best_model_path, map_location=device))
    return model, history, best_val_f1

print('Функції тренування визначено')

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=np.array(y_train)
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

print('Class weights:', class_weights.detach().cpu().numpy())


In [ ]:
print('=' * 60)
print('🅰️  HEAD A: Linear Baseline')
print('=' * 60)

model_a = BertClassifierHeadA(MODEL_NAME, dropout=DROPOUT_RATE).to(device)

total_params_a = sum(p.numel() for p in model_a.parameters())
trainable_a    = sum(p.numel() for p in model_a.parameters() if p.requires_grad)
print(f'Параметрів: {total_params_a:,} (всі trainable: {trainable_a:,})')

model_a, history_a, best_f1_a = train_model(
    model_a, train_loader, val_loader,
    NUM_EPOCHS, LEARNING_RATE, device, model_name='head_a'
)

In [ ]:
# ТРЕНУВАННЯ HEAD B
print('=' * 60)
print('🅱️  HEAD B: MLP + BatchNorm')
print('=' * 60)

model_b = BertClassifierHeadB(MODEL_NAME, dropout=DROPOUT_RATE).to(device)

total_params_b = sum(p.numel() for p in model_b.parameters())
print(f'Параметрів: {total_params_b:,}')

model_b, history_b, best_f1_b = train_model(
    model_b, train_loader, val_loader,
    NUM_EPOCHS, LEARNING_RATE, device, model_name='head_b'
)

In [ ]:
# ТРЕНУВАННЯ HEAD C
print('=' * 60)
print('🅲  HEAD C: Attention Pooling')
print('=' * 60)

model_c = BertClassifierHeadC(MODEL_NAME, dropout=DROPOUT_RATE).to(device)

total_params_c = sum(p.numel() for p in model_c.parameters())
print(f'Параметрів: {total_params_c:,}')

model_c, history_c, best_f1_c = train_model(
    model_c, train_loader, val_loader,
    NUM_EPOCHS, LEARNING_RATE, device, model_name='head_c'
)

In [ ]:
criterion = nn.CrossEntropyLoss()

results = {}
for name, model, history in [
    ('Head A (Linear)', model_a, history_a),
    ('Head B (BN-MLP)', model_b, history_b),
    ('Head C (AttPool)', model_c, history_c)
]:
    loss, acc, f1, y_true, y_pred, y_prob = evaluate(model, test_loader, criterion, device)
    prec = precision_score(y_true, y_pred, average='weighted')
    rec  = recall_score(y_true, y_pred, average='weighted')
    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except:
        roc_auc = None

    results[name] = {
        'acc': acc, 'f1': f1, 'precision': prec, 'recall': rec,
        'roc_auc': roc_auc, 'loss': loss,
        'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_prob,
        'history': history
    }

print('\n' + '=' * 80)
print('ПОРІВНЯННЯ РЕЗУЛЬТАТІВ НА ТЕСТОВОМУ НАБОРІ')
print('=' * 80)
print(f'{"Модель":<20} {"Accuracy":>9} {"F1":>8} {"Precision":>10} {"Recall":>8} {"ROC-AUC":>8}')
print('-' * 80)
for name, r in results.items():
    roc = f"{r['roc_auc']:.4f}" if r['roc_auc'] else 'N/A'
    print(f'{name:<20} {r["acc"]:>9.4f} {r["f1"]:>8.4f} {r["precision"]:>10.4f} {r["recall"]:>8.4f} {roc:>8}')
print('=' * 80)

print('\nПОРІВНЯННЯ З ПОПЕРЕДНІМИ МОДЕЛЯМИ:')
print(f'{"Модель":<35} {"Accuracy":>10} {"F1":>8}')
print('-' * 60)
print(f'{"Logistic Regression + TF-IDF":<35} {"~0.94":>10} {"~0.94":>8}')
print(f'{"LSTM (Bidirectional, 4 layers)":<35} {"~0.50":>10} {"0.67":>8}')
for name, r in results.items():
    print(f'{"BERT " + name:<35} {r["acc"]:>10.4f} {r["f1"]:>8.4f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Динаміка тренування (Train vs Validation)', fontsize=16, fontweight='bold')

colors_map = {
    'Head A (Linear)':  ('#3498db', '#85c1e9'),
    'Head B (BN-MLP)':  ('#e74c3c', '#f1948a'),
    'Head C (AttPool)': ('#2ecc71', '#82e0aa')
}

for col, (name, r) in enumerate(results.items()):
    h = r['history']
    c_dark, c_light = colors_map[name]
    epochs = range(1, len(h['train_loss']) + 1)

    axes[0][col].plot(epochs, h['train_loss'], 'o-', color=c_dark,  label='Train', linewidth=2)
    axes[0][col].plot(epochs, h['val_loss'],   's--', color=c_light, label='Val',   linewidth=2)
    axes[0][col].set_title(f'{name}\nLoss', fontweight='bold')
    axes[0][col].set_xlabel('Epoch')
    axes[0][col].set_ylabel('Loss')
    axes[0][col].legend()
    axes[0][col].grid(alpha=0.3)

    axes[1][col].plot(epochs, h['train_acc'], 'o-', color=c_dark,  label='Train', linewidth=2)
    axes[1][col].plot(epochs, h['val_acc'],   's--', color=c_light, label='Val',   linewidth=2)
    axes[1][col].set_title(f'{name}\nAccuracy', fontweight='bold')
    axes[1][col].set_xlabel('Epoch')
    axes[1][col].set_ylabel('Accuracy')
    axes[1][col].set_ylim([0.5, 1.0])
    axes[1][col].legend()
    axes[1][col].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('bert_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Графік збережено: bert_training_curves.png')

In [ ]:
# CONFUSION MATRIX + ROC CURVES
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Матриці помилок та ROC-криві', fontsize=16, fontweight='bold')

for col, (name, r) in enumerate(results.items()):
    c_dark = list(colors_map.values())[col][0]

    # Confusion Matrix
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    sns.heatmap(
        cm, annot=True, fmt='d', ax=axes[0][col],
        cmap='Blues',
        xticklabels=['FAKE', 'TRUE'],
        yticklabels=['FAKE', 'TRUE'],
        linewidths=1, linecolor='white'
    )
    axes[0][col].set_title(f'{name}\nConfusion Matrix (Acc={r["acc"]:.3f})', fontweight='bold')
    axes[0][col].set_xlabel('Передбачено')
    axes[0][col].set_ylabel('Реальне')

    # ROC Curve
    if r['roc_auc']:
        fpr, tpr, _ = roc_curve(r['y_true'], r['y_prob'])
        axes[1][col].plot(fpr, tpr, color=c_dark, lw=2,
                          label=f'ROC (AUC = {r["roc_auc"]:.3f})')
    axes[1][col].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
    axes[1][col].set_title(f'{name}\nROC Curve', fontweight='bold')
    axes[1][col].set_xlabel('False Positive Rate')
    axes[1][col].set_ylabel('True Positive Rate')
    axes[1][col].legend()
    axes[1][col].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('bert_confusion_roc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Графік збережено: bert_confusion_roc.png')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Порівняння всіх підходів', fontsize=16, fontweight='bold')

all_models = {
    'LogReg\nTF-IDF':   {'acc': 0.94, 'f1': 0.94},
    'LSTM\n(BiDir)':    {'acc': 0.50, 'f1': 0.67},
    'BERT\nHead A':     {'acc': results['Head A (Linear)']['acc'],  'f1': results['Head A (Linear)']['f1']},
    'BERT\nHead B':     {'acc': results['Head B (BN-MLP)']['acc'],  'f1': results['Head B (BN-MLP)']['f1']},
    'BERT\nHead C':     {'acc': results['Head C (AttPool)']['acc'], 'f1': results['Head C (AttPool)']['f1']},
}

model_names = list(all_models.keys())
accs  = [v['acc'] for v in all_models.values()]
f1s   = [v['f1']  for v in all_models.values()]
bar_colors = ['#95a5a6', '#e74c3c', '#3498db', '#e67e22', '#2ecc71']

for ax, metric, values, title in [
    (axes[0], 'Accuracy', accs, 'Accuracy по моделях'),
    (axes[1], 'F1-Score', f1s,  'F1-Score по моделях')
]:
    bars = ax.bar(model_names, values, color=bar_colors, alpha=0.85, edgecolor='black', linewidth=1.2)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_ylim([0.4, 1.05])
    ax.axhline(y=0.94, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='LogReg baseline (0.94)')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('bert_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Графік збережено: bert_comparison.png')

In [ ]:
# ДЕТАЛЬНИЙ CLASSIFICATION REPORT
best_name = max(results, key=lambda k: results[k]['f1'])
best_r = results[best_name]

print(f'Найкраща модель: {best_name}')
print('=' * 60)
print(classification_report(
    best_r['y_true'], best_r['y_pred'],
    target_names=['FAKE', 'TRUE'],
    digits=4
))

print('\nClassification Report для всіх моделей:')
for name, r in results.items():
    print(f'\n--- {name} ---')
    print(classification_report(
        r['y_true'], r['y_pred'],
        target_names=['FAKE', 'TRUE'],
        digits=4
    ))

In [ ]:
def predict_news(text, model, tokenizer, max_len=MAX_LEN, device=device):
    model.eval()

    encoding = tokenizer(
        text,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()

    label = 'TRUE ✅' if probs[1] > 0.5 else 'FAKE ❌'
    confidence = max(probs)

    return {
        'text':       text[:100] + '...' if len(text) > 100 else text,
        'label':      label,
        'confidence': confidence,
        'prob_true':  probs[1],
        'prob_fake':  probs[0]
    }


def print_prediction(result):
    print('\n' + '=' * 70)
    print('РЕЗУЛЬТАТ ПЕРЕДБАЧЕННЯ (BERT)')
    print('=' * 70)
    print(f'\nТекст: {result["text"]}')
    print(f'\nПередбачення: {result["label"]}')
    print(f'Впевненість: {result["confidence"]*100:.2f}%')
    print(f'\nЙмовірності:')
    print(f'   TRUE: {result["prob_true"]*100:.2f}%')
    print(f'   FAKE: {result["prob_fake"]*100:.2f}%')
    print('=' * 70)

print('Функція передбачення визначена')

In [ ]:
best_name = max(results, key=lambda k: results[k]['f1'])
best_model_map = {
    'Head A (Linear)':  model_a,
    'Head B (BN-MLP)':  model_b,
    'Head C (AttPool)': model_c
}
best_model = best_model_map[best_name]
print(f'Використовуємо: {best_name}\n')

# Тестові новини
test_news = [
    # Фейкова
    "Компанія Apple анонсувала розробку нового смартфона, який заряджається виключно від звукових хвиль під час розмови власника.",
    # Правдива
    "Верховна Рада України прийняла закон про державний бюджет на 2024 рік у другому читанні.",
    # Очевидний фейк
    "Вчені довели що земля пласка і НАСА приховує це вже 50 років! Терміново поширте!",
]

for i, text in enumerate(test_news, 1):
    print(f'\n--- Приклад {i} ---')
    result = predict_news(text, best_model, tokenizer)
    print_prediction(result)

In [ ]:
best_acc  = results[best_name]['acc']
best_f1   = results[best_name]['f1']

torch.save({
    'model_state_dict': best_model.state_dict(),
    'model_name':       MODEL_NAME,
    'head_type':        best_name,
    'max_len':          MAX_LEN,
    'test_accuracy':    best_acc,
    'test_f1':          best_f1,
    'all_results':      {k: {m: v for m, v in r.items() if m not in ['y_true', 'y_pred', 'y_prob', 'history']} for k, r in results.items()}
}, 'bert_fake_detector_best.pt')

tokenizer.save_pretrained('bert_tokenizer/')

print('\n' + '=' * 70)
print('ЗБЕРЕЖЕННЯ ЗАВЕРШЕНО')
print('=' * 70)
print(f'bert_fake_detector_best.pt — найкраща модель ({best_name})')
print(f'   Accuracy: {best_acc:.4f} | F1: {best_f1:.4f}')
print(f'bert_tokenizer/ — токенізатор')
print('=' * 70)

In [ ]:
print('\n' + '=' * 80)
print('ПІДСУМКОВА ТАБЛИЦЯ ВСІХ МОДЕЛЕЙ')
print('=' * 80)

summary_rows = [
    ('Logistic Regression + TF-IDF', '~0.9400', '~0.9400', '~0.9400', '~0.9400', 'N/A',    'Базовий'),
    ('LSTM (Bidirectional, 4 L.)',    '~0.5000', '~0.6667', '~0.5000', '~1.0000', '~0.583', 'Деградував'),
]

for name, r in results.items():
    roc = f"{r['roc_auc']:.4f}" if r['roc_auc'] else 'N/A'
    flag = 'НАЙКРАЩЕ' if name == best_name else ''
    summary_rows.append((
        f'BERT {name}',
        f"{r['acc']:.4f}",
        f"{r['f1']:.4f}",
        f"{r['precision']:.4f}",
        f"{r['recall']:.4f}",
        roc,
        flag
    ))

print(f'{"Модель":<35} {"Acc":>7} {"F1":>7} {"Prec":>7} {"Rec":>7} {"AUC":>7}  Статус')
print('-' * 80)
for row in summary_rows:
    print(f'{row[0]:<35} {row[1]:>7} {row[2]:>7} {row[3]:>7} {row[4]:>7} {row[5]:>7}  {row[6]}')

print('=' * 80)
print(f'\n✅ Найкраща архітектура: {best_name}')
print(f'   Accuracy: {results[best_name]["acc"]:.4f}')
print(f'   F1-Score: {results[best_name]["f1"]:.4f}')